In [ ]:
import os, subprocess, re, shutil
from pathlib import Path

from google.colab import drive
drive.mount('/content/drive')

WORK_DIR = "/content/SeaS"
if not os.path.exists(WORK_DIR):
    !git clone https://github.com/HUST-SLOW/SeaS.git {WORK_DIR}
os.chdir(WORK_DIR)
print(f"작업 디렉토리: {os.getcwd()}")

if not os.path.exists('/content/miniconda'):
    !wget -q https://repo.anaconda.com/miniconda/Miniconda3-latest-Linux-x86_64.sh
    !bash Miniconda3-latest-Linux-x86_64.sh -b -p /content/miniconda
    !rm Miniconda3-latest-Linux-x86_64.sh

os.environ['PATH'] = '/content/miniconda/bin:' + os.environ.get('PATH', '')
!conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/main -q

!conda create -n seas python=3.9 -y -q
!conda run -n seas pip install torch==2.2.1 torchvision==0.17.1 --index-url https://download.pytorch.org/whl/cu118 -q
!conda run -n seas pip install huggingface_hub==0.23.4 transformers==4.30.0 accelerate==0.20.3 numpy==1.26.4 scikit-learn pandas Pillow tqdm matplotlib scipy einops bitsandbytes==0.41.0 -q

os.makedirs(f"{WORK_DIR}/configs", exist_ok=True)
with open(f"{WORK_DIR}/configs/accelerater_config.yaml", "w") as f:
    f.write("compute_environment: LOCAL_MACHINE\ndistributed_type: 'NO'\ndowncast_bf16: 'no'\ngpu_ids: '0'\nmachine_rank: 0\nmain_training_function: main\nmixed_precision: 'no'\nnum_machines: 1\nnum_processes: 1\nrdzv_backend: static\nsame_network: true\ntpu_env:[]\ntpu_use_cluster: false\ntpu_use_sudo: false\nuse_cpu: false\n")

seas_config = """training:
  datasets:
    resolution: 512
    num_normal_images: 30
    rotation: False
    dataloader_num_workers: 0
    batch_size:
      gen_train_batch_size: 1
      mask_train_batch_size: 1
  models:
    pretrained_model_name_or_path: ./model_hub/stable-diffusion-v1-4
    prompt:
      normal_token_num: 1
      anomaly_token_num: 4
      initializer_token: "defect"
  optimizer:
    gen_learning_rate: 0.000004
    rmp_learning_rate: 0.0005
    encoder_learning_rate: 0.00004
    lr_scheduler: constant
    lr_warmup_steps: 0
    gradient_accumulation_steps: 1
    adam_beta1: 0.9
    adam_beta2: 0.999
    adam_weight_decay: 0.01
    adam_epsilon1e-08: 1e-08
    lr_num_cycles: 1
    lr_power: 1.0
    use_8bit_adam: True
  train_text_encoder: True
  with_Ni_Alignment: True
  mixed_precision: "fp16"
  offset_noise: False
inference:
  datasets:
    batch_size: 1
    add_noise_step: 1500
  models:
    stable_diffusion_model_path: ./model_hub/stable-diffusion-v1-4
    num_inference_steps: 25
    guidance_scale: 2
  seed_start: 0
  threshold: 0.2
  onlyfinal: True
  gen_mask: True
  device: "cuda:0"
"""
with open(f"{WORK_DIR}/configs/seas.yaml", "w") as f:
    f.write(seas_config)

seas_py_path = f"{WORK_DIR}/models/seas.py"
with open(seas_py_path, "r") as f: content = f.read()
new_content = re.sub(r'shutil\.copytree\((\w+),\s*(\w+)\)', r'shutil.copytree(\1, \2, dirs_exist_ok=True)', content)
with open(seas_py_path, "w") as f: f.write(new_content)

seas_mask_path = f"{WORK_DIR}/models/seas_mask.py"
with open(seas_mask_path, "r") as f: content = f.read()

content = content.replace("timesteps = torch.randint(0, noise_scheduler.config.num_train_timesteps, (4,), device=accelerator.device)",
                          "timesteps = torch.randint(0, noise_scheduler.config.num_train_timesteps, (pixel_values.shape[0],), device=accelerator.device)")
content = content.replace("unet.to(accelerator.device, dtype=torch.float32)", "unet.to(accelerator.device, dtype=weight_dtype)")
content = content.replace("accelerator.log(logs, step=global_step)", "pass")
content = content.replace("accelerator.end_training()", "pass")
content = content.replace("os.makedirs(save_path)", "os.makedirs(save_path, exist_ok=True)")
content = content.replace("torch.save(rmp.state_dict(), os.path.join(save_path, sub_dir))",
"""import shutil
                            rmp_target = os.path.join(save_path, sub_dir)
                            if os.path.isdir(rmp_target): shutil.rmtree(rmp_target)
                            torch.save(accelerator.unwrap_model(rmp).state_dict(), rmp_target)""")

with open(seas_mask_path, "w") as f: f.write(content)

from huggingface_hub import snapshot_download
MODEL_DIR = f"{WORK_DIR}/model_hub/stable-diffusion-v1-4"
os.makedirs(MODEL_DIR, exist_ok=True)
if not os.path.exists(os.path.join(MODEL_DIR, "model_index.json")):
    snapshot_download(repo_id="CompVis/stable-diffusion-v1-4", local_dir=MODEL_DIR, ignore_patterns=["*.msgpack", "*.h5", "flax_model*"])

print("\nPart 1 (초기 셋업 및 패치) 완벽하게 완료!")

In [ ]:
import os, shutil
import pandas as pd
from pathlib import Path
from sklearn.model_selection import train_test_split
from PIL import Image

DRIVE_DATASET_ROOT = "/content/drive/MyDrive/Dataset/ViSA dataset"
SEAS_DATA_ROOT     = "/content/SeaS/data/visa"
PCB_CATEGORIES     =["pcb1", "pcb2", "pcb3", "pcb4"]
NORMAL_TRAIN_RATIO = 0.8
RANDOM_SEED        = 42

def is_single_defect(label):
    return ',' not in str(label) and label != 'normal'

def convert(category):
    src_root    = Path(DRIVE_DATASET_ROOT) / category
    dst_root    = Path(SEAS_DATA_ROOT) / category
    src_normal  = src_root / "Data" / "Images" / "Normal"
    src_anomaly = src_root / "Data" / "Images" / "Anomaly"
    src_masks   = src_root / "Data" / "Masks"  / "Anomaly"

    print(f"\n처리 중: {category}")

    df = pd.read_csv(src_root / "image_anno.csv")
    df_single = df[(df['label'] != 'normal') & (df['label'].apply(is_single_defect))]

    defect_type_map = {}
    for _, row in df_single.iterrows():
        stem  = Path(str(row['image'])).stem
        dtype = str(row['label']).lower().replace(" ", "_")
        defect_type_map[stem] = dtype

    exts =["*.jpg", "*.JPG", "*.png", "*.PNG"]
    normal_imgs = [f for e in exts for f in src_normal.glob(e)]
    train_imgs, test_good_imgs = train_test_split(
        normal_imgs, train_size=NORMAL_TRAIN_RATIO, random_state=RANDOM_SEED)

    def copy_files(file_list, dst_dir):
        dst_dir.mkdir(parents=True, exist_ok=True)
        for f in file_list:
            dst = dst_dir / f.name
            if not dst.exists():
                shutil.copy2(str(f), str(dst))

    copy_files(train_imgs,     dst_root / "train" / "good")
    copy_files(test_good_imgs, dst_root / "test"  / "good")

    anomaly_imgs  = sorted([f for e in exts for f in src_anomaly.glob(e)])
    anomaly_masks = sorted([f for e in exts for f in src_masks.glob(e)])
    mask_map      = {f.stem: f for f in anomaly_masks}

    copied_imgs = copied_masks = 0
    for img_path in anomaly_imgs:
        stem = img_path.stem
        if stem not in defect_type_map:
            continue
        dtype = defect_type_map[stem]

        dst_img_dir = dst_root / "test" / dtype
        dst_img_dir.mkdir(parents=True, exist_ok=True)
        existing = sorted(dst_img_dir.glob("*.png"))
        new_name = f"{len(existing):03}.png"
        img = Image.open(str(img_path)).convert("RGB")
        img.save(str(dst_img_dir / new_name))
        copied_imgs += 1

        if stem in mask_map:
            dst_mask_dir = dst_root / "ground_truth" / dtype
            dst_mask_dir.mkdir(parents=True, exist_ok=True)
            existing_masks = sorted(dst_mask_dir.glob("*.png"))
            mask_name = f"{len(existing_masks):03}.png"
            shutil.copy2(str(mask_map[stem]), str(dst_mask_dir / mask_name))
            copied_masks += 1

    print(f"  정상: train {len(train_imgs)}장 / test {len(test_good_imgs)}장")
    print(f"  결함 이미지: {copied_imgs}장 / 마스크: {copied_masks}장")

if Path(SEAS_DATA_ROOT).exists():
    shutil.rmtree(SEAS_DATA_ROOT)

for cat in PCB_CATEGORIES:
    convert(cat)

print("\nPart 2 완료")

In [ ]:
import os, subprocess, shutil
from pathlib import Path

os.environ['PATH'] = '/content/miniconda/bin:' + os.environ.get('PATH', '')
WORK_DIR       = "/content/SeaS"
SEAS_DATA_ROOT = "/content/SeaS/data/visa"
PCB_CATEGORIES =["pcb1", "pcb2", "pcb3", "pcb4"]

GEN_TRAIN_STEPS     = 1000
GEN_CHECKPOINT_STEP = 100
MASK_TRAIN_STEPS    = 300
MASK_CHECKPOINT_STEP= 50

def get_defect_types(category):
    test_dir = Path(SEAS_DATA_ROOT) / category / "test"
    if not test_dir.exists(): return []
    return sorted([d.name for d in test_dir.iterdir() if d.is_dir() and d.name != "good"])

failed =[]

for category in PCB_CATEGORIES:
    defect_types = get_defect_types(category)
    if not defect_types: continue

    print(f"\n{category}: {defect_types}")
    for defect_type in defect_types:
        output_dir = f"{WORK_DIR}/outputs/{category}/{defect_type}"

        print(f"\n[1/2] Generation 학습: {category} / {defect_type}")
        shutil.rmtree(output_dir, ignore_errors=True)

        cmd_gen =[
            "conda", "run", "-n", "seas", "accelerate", "launch",
            "--config_file", f"{WORK_DIR}/configs/accelerater_config.yaml",
            f"{WORK_DIR}/examples/SeaS_main.py",
            "--pretrained_model_name_or_path", f"{WORK_DIR}/model_hub/stable-diffusion-v1-4",
            "--output_dir",          output_dir,
            "--instance_data_dir",   f"{SEAS_DATA_ROOT}/{category}/test",
            "--mask_dir",            f"{SEAS_DATA_ROOT}/{category}/ground_truth",
            "--normal_data_dir",     f"{SEAS_DATA_ROOT}/{category}/train/good",
            "--gen_train_steps",     str(GEN_TRAIN_STEPS),
            "--checkpointing_steps", str(GEN_CHECKPOINT_STEP),
            "--mixed_precision",     "fp16", "--use_8bit_adam",
            "--config",              f"{WORK_DIR}/configs/seas.yaml",
        ]
        res_gen = subprocess.run(cmd_gen, capture_output=True, text=True, cwd=WORK_DIR, env={**os.environ, 'MPLBACKEND': 'Agg'})
        if res_gen.returncode != 0:
            print(f"Gen 실패: {res_gen.stderr[-500:]}"); failed.append(f"{category}/{defect_type} (Gen)"); continue

        print(f"[2/2] Mask 학습: {category} / {defect_type}")
        cmd_mask =[
            "conda", "run", "-n", "seas", "accelerate", "launch",
            "--config_file", f"{WORK_DIR}/configs/accelerater_config.yaml",
            f"{WORK_DIR}/examples/SeaS_mask_main.py",
            "--pretrained_model_name_or_path", f"{WORK_DIR}/model_hub/stable-diffusion-v1-4",
            "--output_dir",                output_dir,
            "--instance_data_dir",         f"{SEAS_DATA_ROOT}/{category}/test",
            "--mask_dir",                  f"{SEAS_DATA_ROOT}/{category}/ground_truth",
            "--normal_data_dir",           f"{SEAS_DATA_ROOT}/{category}/train/good",
            "--seas_trained_model_path",   f"{output_dir}/generation-checkpoint",
            "--mask_train_steps",          str(MASK_TRAIN_STEPS),
            "--checkpointing_steps",       str(MASK_CHECKPOINT_STEP),
            "--mixed_precision",           "fp16", "--use_8bit_adam",
            "--config",                    f"{WORK_DIR}/configs/seas.yaml",
        ]
        res_mask = subprocess.run(cmd_mask, capture_output=True, text=True, cwd=WORK_DIR, env={**os.environ, 'MPLBACKEND': 'Agg'})
        if res_mask.returncode != 0:
            print(f"Mask 실패: {res_mask.stderr[-500:]}"); failed.append(f"{category}/{defect_type} (Mask)")
        else:
            print(f"학습 완료: {category}/{defect_type}")

print("\n" + "="*60)
print(f"실패: {failed}" if failed else "전체 학습 완벽하게 완료!")

In [ ]:
import os, subprocess, shutil, json, csv
from pathlib import Path
from PIL import Image
import numpy as np

os.environ['PATH'] = '/content/miniconda/bin:' + os.environ.get('PATH', '')
WORK_DIR        = "/content/SeaS"
SEAS_DATA_ROOT  = "/content/SeaS/data/visa"
DRIVE_OUTPUT    = "/content/drive/MyDrive/SeaS_Results"
PCB_CATEGORIES  =["pcb1", "pcb2", "pcb3", "pcb4"]
TOTAL_INFER_NUM = 50

os.makedirs(DRIVE_OUTPUT, exist_ok=True)

def get_defect_types(category):
    test_dir = Path(SEAS_DATA_ROOT) / category / "test"
    if not test_dir.exists(): return []
    return sorted([d.name for d in test_dir.iterdir() if d.is_dir() and d.name != "good"])

def make_prompt(normal_token_num=1, anomaly_token_num=4):
    ob_tokens  = " ".join([f"ob{i}" for i in range(1, normal_token_num + 1)])
    sks_tokens = " ".join([f"sks{i}" for i in range(1, anomaly_token_num + 1)])
    return f"a {ob_tokens} with {sks_tokens}"

def calc_metrics(generated_image_dir: Path, real_dir: Path) -> dict:
    if generated_image_dir.exists():
        gen_imgs = list(generated_image_dir.glob("*.png")) + list(generated_image_dir.glob("*.jpg"))
    else:
        gen_imgs =[]

    real_imgs = list(real_dir.glob("*.png")) + list(real_dir.glob("*.jpg"))

    def img_stats(img_paths):
        brightness_list = []
        for p in img_paths[:50]:
            try:
                img = np.array(Image.open(str(p)).convert("RGB")).astype(float)
                brightness_list.append(img.mean())
            except: pass
        if not brightness_list: return 0.0, 0.0
        return float(np.mean(brightness_list)), float(np.std(brightness_list))

    gen_mean, gen_std   = img_stats(gen_imgs)
    real_mean, real_std = img_stats(real_imgs)

    return {
        "generated_count":  len(gen_imgs),
        "real_count":       len(real_imgs),
        "gen_brightness_mean":  round(gen_mean,  2),
        "gen_brightness_std":   round(gen_std,   2),
        "real_brightness_mean": round(real_mean, 2),
        "real_brightness_std":  round(real_std,  2),
        "brightness_diff":      round(abs(gen_mean - real_mean), 2),
    }

all_metrics = {}
failed =[]

for category in PCB_CATEGORIES:
    defect_types = get_defect_types(category)
    if not defect_types: continue

    print(f"\n{category}: {defect_types}")
    all_metrics[category] = {}

    for defect_type in defect_types:
        gen_ckpt = f"{WORK_DIR}/outputs/{category}/{defect_type}/generation-checkpoint"
        rmp_ckpt = f"{WORK_DIR}/outputs/{category}/{defect_type}/mask-checkpoint/rmp"

        if not Path(gen_ckpt).exists() or not Path(rmp_ckpt).exists():
            print(f"  체크포인트 없음: {category}/{defect_type}")
            continue

        output_dir = f"{WORK_DIR}/outputs/images/{category}/{defect_type}"
        prompt = make_prompt()

        cmd =[
            "conda", "run", "-n", "seas", "python", f"{WORK_DIR}/examples/SeaS_infer.py",
            "--output_dir",      output_dir,
            "--ref_data_dir",    f"{SEAS_DATA_ROOT}/{category}/train/good",
            "--gen_model_path",  gen_ckpt,
            "--rmp_model_path",  rmp_ckpt,
            "--prompt",          prompt,
            "--total_infer_num", str(TOTAL_INFER_NUM),
            "--config",          f"{WORK_DIR}/configs/seas.yaml",
            "--stable_diffusion_model_path", f"{WORK_DIR}/model_hub/stable-diffusion-v1-4",
        ]

        print(f"Inference 진행 중: {category} / {defect_type}")
        result = subprocess.run(cmd, capture_output=True, text=True, cwd=WORK_DIR, env={**os.environ, 'MPLBACKEND': 'Agg'})

        if result.returncode != 0:
            print(f"  실패: {result.stderr[-500:]}"); failed.append(f"{category}/{defect_type}"); continue

        gen_image_dir = Path(output_dir) / "image"
        real_dir      = Path(SEAS_DATA_ROOT) / category / "test" / defect_type
        metrics       = calc_metrics(gen_image_dir, real_dir)
        all_metrics[category][defect_type] = metrics

        print(f"  이미지 생성: {metrics['generated_count']}장 | 밝기 차이: {metrics['brightness_diff']:.2f}")

        drive_dest = Path(DRIVE_OUTPUT) / category / defect_type
        shutil.copytree(output_dir, str(drive_dest), dirs_exist_ok=True)

with open(f"{DRIVE_OUTPUT}/metrics.json", "w") as f:
    json.dump(all_metrics, f, indent=2, ensure_ascii=False)

with open(f"{DRIVE_OUTPUT}/metrics.csv", "w", newline='') as f:
    writer = csv.writer(f)
    writer.writerow(["category", "defect_type", "generated_count", "real_count", "gen_mean", "gen_std", "real_mean", "real_std", "diff"])
    for cat, defects in all_metrics.items():
        for d_type, m in defects.items():
            writer.writerow([cat, d_type, m["generated_count"], m["real_count"], m["gen_brightness_mean"], m["gen_brightness_std"], m["real_brightness_mean"], m["real_brightness_std"], m["brightness_diff"]])

print("\n" + "="*60)
print(f"실패: {failed}" if failed else "전체 Inference 완벽하게 완료!")
print(f"Drive 확인: {DRIVE_OUTPUT}")

In [ ]:
!pip install torchmetrics[image] scikit-image -q

import os
import torch
import numpy as np
import pandas as pd
from pathlib import Path
from PIL import Image
from tqdm import tqdm
from skimage.metrics import structural_similarity as ssim

from torchmetrics.image.fid import FrechetInceptionDistance
from torchmetrics.image.kid import KernelInceptionDistance

WORK_DIR        = "/content/SeaS"
SEAS_DATA_ROOT  = "/content/SeaS/data/visa"
DRIVE_OUTPUT    = "/content/drive/MyDrive/SeaS_Results"
PCB_CATEGORIES  =["pcb1", "pcb2", "pcb3", "pcb4"]

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def load_images_as_tensor(image_paths, size=(299, 299)):
    """이미지를 읽어서 FID/KID 계산을 위한 텐서(Uint8, RGB)로 변환"""
    imgs =[]
    for p in image_paths:
        try:
            img = Image.open(str(p)).convert("RGB").resize(size)
            img_np = np.array(img)

            img_tensor = torch.tensor(img_np).permute(2, 0, 1)
            imgs.append(img_tensor)
        except Exception as e:
            pass
    if not imgs:
        return torch.empty(0)
    return torch.stack(imgs)

def calc_ssim_score(real_imgs_paths, gen_imgs_paths):
    """실제 이미지와 생성 이미지 간의 평균 구조적 유사도(SSIM) 계산"""
    if not real_imgs_paths or not gen_imgs_paths:
        return 0.0

    score_list =[]

    for g_path in gen_imgs_paths:
        r_path = np.random.choice(real_imgs_paths)
        try:
            g_img = np.array(Image.open(str(g_path)).convert("L").resize((256, 256)))
            r_img = np.array(Image.open(str(r_path)).convert("L").resize((256, 256)))


            val = ssim(r_img, g_img, data_range=255)
            score_list.append(val)
        except:
            pass
    return float(np.mean(score_list)) if score_list else 0.0

results =[]

for category in PCB_CATEGORIES:
    test_dir = Path(SEAS_DATA_ROOT) / category / "test"
    if not test_dir.exists(): continue

    defect_types = sorted([d.name for d in test_dir.iterdir() if d.is_dir() and d.name != "good"])

    for defect_type in defect_types:
        print(f"\n평가 진행 중: {category} / {defect_type} ...")

        real_dir = test_dir / defect_type
        real_paths = list(real_dir.glob("*.png")) + list(real_dir.glob("*.jpg"))


        gen_dir = Path(DRIVE_OUTPUT) / category / defect_type / "image"
        gen_paths = list(gen_dir.glob("*.png")) + list(gen_dir.glob("*.jpg"))

        if len(real_paths) < 2 or len(gen_paths) < 2:
            print(f"  이미지가 부족하여 평가를 건너뜁니다. (최소 2장 이상 필요)")
            continue


        real_tensors = load_images_as_tensor(real_paths)
        gen_tensors = load_images_as_tensor(gen_paths)


        fid = FrechetInceptionDistance(feature=64).to(device)
        kid = KernelInceptionDistance(subset_size=min(len(real_tensors), len(gen_tensors), 50)).to(device)

        fid.update(real_tensors.to(device), real=True)
        fid.update(gen_tensors.to(device), real=False)
        fid_score = float(fid.compute())

        kid.update(real_tensors.to(device), real=True)
        kid.update(gen_tensors.to(device), real=False)
        kid_mean, kid_std = kid.compute()
        kid_score = float(kid_mean)


        ssim_score = calc_ssim_score(real_paths, gen_paths)

        print(f"  FID: {fid_score:.2f} | KID: {kid_score:.4f} | SSIM: {ssim_score:.4f}")

        results.append({
            "Category": category,
            "Defect_Type": defect_type,
            "FID (↓)": round(fid_score, 2),
            "KID (↓)": round(kid_score, 4),
            "SSIM (↑)": round(ssim_score, 4)
        })


df_results = pd.DataFrame(results)
csv_save_path = f"{DRIVE_OUTPUT}/evaluation_metrics.csv"
df_results.to_csv(csv_save_path, index=False)

print("\n" + "="*60)
print("최종 평가 지표 계산 완료!")
print(f"결과 저장됨: {csv_save_path}")
print("\n[평가 결과 요약]")
print(df_results.to_string(index=False))